<a href="https://colab.research.google.com/github/dhanush435/BankingApplication/blob/main/EMOSENSE_MINOR2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python pydub moviepy SpeechRecognition deepface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.5 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=8d97183a5cd5738a30a03b29d33bf01675b77254be780cf97684a335336c9c80
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire


In [ ]:
# @title MODULE 1: VIDEO TO EMOTION DETECTION

video_path = "/content/emo-vid1.mp4"
import cv2
from deepface import DeepFace
from google.colab.patches import cv2_imshow

# Load the face detection model
face_model = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
frame_list = []

# Load the video
capture = cv2.VideoCapture(video_path)
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
print("Total number of frames in the video:", total_frames)

frame_skip = 5  # Process every 5th frame
resize_factor = 0.5  # Resize factor for reducing frame resolution

for i in range(0, total_frames, frame_skip):
    ret, frame = capture.read()
    if not ret:
        break

    # Resize the frame to speed up processing
    frame = cv2.resize(frame, (0, 0), fx=resize_factor, fy=resize_factor)

    # Detect faces
    face = face_model.detectMultiScale(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 1.1, 5)

    for x, y, width, height in face:
        # Analyze emotion
        emotion = DeepFace.analyze(frame, actions=["emotion"], enforce_detection=False)

        # Draw the dominant emotion on the frame
        cv2.putText(frame, str(emotion[0]["dominant_emotion"]), (x, y + height),
                    cv2.FONT_HERSHEY_COMPLEX, 0.9, (255, 255, 0), 2)
        cv2.rectangle(frame, (x, y), (x + width, y + height), (255, 255, 0), 2)
        frame_list.append(frame)

# Release the video capture
capture.release()

# Write the output video
output_path = "Emotions.avi"
output = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"DIVX"), 60, (int(frame.shape[1]), int(frame.shape[0])))
for frame in frame_list:
    output.write(frame)

output.release()

print("Emotion detection completed and video saved successfully!")


Total number of frames in the video: 0


NameError: name 'frame' is not defined

In [ ]:
!pip install moviepy pydub


In [ ]:
# @title MODULE 1:VIDEO TO AUDIO DETECTION
from moviepy.editor import VideoFileClip

# Specify the video file path
video_path = "/content/emo-vid1.mp4"  # Replace with your video file path
audio_path = "output_audio.mp3"  # Specify the output audio file path

# Load the video file
video_clip = VideoFileClip(video_path)

# Extract audio and write to file
audio_clip = video_clip.audio
audio_clip.write_audiofile(audio_path)

# Close the clips
audio_clip.close()
video_clip.close()

print("Audio extracted successfully!")


MoviePy - Writing audio in output_audio.mp3


MoviePy - Done.
Audio extracted successfully!


In [ ]:
!pip install SpeechRecognition pydub


In [ ]:
# @title MODULE 1:AUDIO TO TEXT DETECTION
import speech_recognition as sr
from pydub import AudioSegment

# Specify the audio file path
audio_path = "output_audio.mp3"  # Use the audio file generated in the previous step

# Convert mp3 to wav format (SpeechRecognition works best with wav)
audio = AudioSegment.from_mp3(audio_path)
wav_audio_path = "output_audio.wav"
audio.export(wav_audio_path, format="wav")

# Initialize recognizer
recognizer = sr.Recognizer()

# Load the audio file
with sr.AudioFile(wav_audio_path) as source:
    audio_data = recognizer.record(source)

# Recognize speech using Google Web Speech API
try:
    text = recognizer.recognize_google(audio_data)
    print("Transcribed Text: ", text)
    +
except sr.UnknownValueError:
    print("Could not understand audio")
except sr.RequestError as e:
    print(f"Could not request results from Google Speech Recognition service; {e}")



Transcribed Text:  feel like I'm drowning in my own thoughts and memories it's like a never-ending nightmare


In [ ]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# @title KEY WORDS FINDING
from nltk.tokenize import word_tokenize
from collections import Counter
text = "I feel like I'm drowning in my own thoughts and memories. It's like a never-ending nightmare."

# Function to summarize the text briefly
def summarize_text(text):
  words = text.split()
  summary_words = [word for word in words if len(word) > 4]  # Filter out short words
  summary = " ".join(summary_words[:10])  # Take the first 10 words
  return summary

# Function to find common keywords
def find_common_keywords(text, keywords):
  text_words = set(text.lower().split())
  common_keywords = text_words.intersection(set(keywords))
  return common_keywords

# Summarize the text
summary = summarize_text(text)
print("Summary:", summary)

# Find common keywords
mental_health_keywords = [
     "depression", "anxiety", "bipolar disorder", "schizophrenia", "obsessive-compulsive disorder (OCD)",
    "post-traumatic stress disorder (PTSD)", "eating disorders (e.g., anorexia, bulimia)",
    "attention-deficit/hyperactivity disorder (ADHD)", "autism spectrum disorder", "personality disorders",
    "substance abuse", "addiction", "self-harm", "suicidal thoughts", "stress", "trauma",
    "coping mechanisms", "counseling", "psychotherapy", "medication"
]

tokens = word_tokenize(text.lower())

# Count the occurrences of each keyword in the text
keyword_counts = Counter(tokens)

# Print common keywords and their counts
for keyword, count in keyword_counts.items():
    if keyword in  mental_health_keywords:
        print(keyword, ":", count)


Summary: drowning thoughts memories. never-ending nightmare.


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
import re

# Load the detected emotions from the video file
video_emotions = ['Neutral']  # Example emotions detected from the video

# Example text containing detected speech
detected_text = """
drowning thoughts memories. never-ending nightmare.
"""

# Function to analyze text for emotional cues
def analyze_text(text):
    # Example keywords for different emotions
    emotion_keywords = {
        'sadness': ['sad', 'unhappy', 'hopeless', 'down', 'depressed',],
        'anger': ['angry', 'furious', 'frustrated', 'hate', 'rage'],
        'fear': ['fear', 'scared', 'anxious', 'worried', 'terrified','drowning','nightmare'],
        'joy': ['happy', 'joyful', 'excited', 'elated', 'pleased']
    }

    detected_emotions = []

    for emotion, keywords in emotion_keywords.items():
        # Check if any of the keywords are present in the text
        if any(re.search(r'\b' + re.escape(keyword) + r'\b', text, re.IGNORECASE) for keyword in keywords):
            detected_emotions.append(emotion)

    return detected_emotions

# Analyze the text for emotional cues
text_emotions = analyze_text(detected_text)

# Combine emotions from video and text
combined_emotions = list(set(video_emotions + text_emotions))

# Define a function to map emotions to potential mental health disorders
def map_emotions_to_disorders(emotions):
    # Define rules or thresholds to map detected emotions to potential disorders
    disorder_map = {
        'sadness': ['Depression'],
        'anger': ['Anger Management Issues', 'Intermittent Explosive Disorder'],
        'fear': ['Anxiety Disorders', 'Panic Disorder'],
        'joy': ['Bipolar Disorder (Manic Phase)'],  # Can indicate mania in some contexts
    }

    potential_disorders = set()

    for emotion in emotions:
        if emotion in disorder_map:
            potential_disorders.update(disorder_map[emotion])

    return list(potential_disorders)

# Call the function to map emotions to potential disorders
potential_disorders = map_emotions_to_disorders(combined_emotions)

# Output potential disorders
print("Potential mental health disorders:")
for disorder in potential_disorders:
    print(disorder)


Potential mental health disorders:
Anxiety Disorders
Anger Management Issues
Panic Disorder
Intermittent Explosive Disorder
